# AI Inventory Intelligence & Supplier Orchestration 📦🛡️
Run the full Streamlit Dashboard directly from Google Colab!

**Instructions:**
1. Run the first cell to setup the environment.
2. Add your API keys in the second cell.
3. Run the final cell to launch. **No authentication required!**

In [ ]:
!git clone https://github.com/lmudu2/ai-inventory-intelligence.git
%cd ai-inventory-intelligence
!pip install streamlit pandas plotly langgraph langchain-groq langchain-core groq python-dotenv requests numpy sendgrid

In [ ]:
import os
# ⚠️ ENTER YOUR STAGING API KEYS HERE ⚠️
os.environ['GROQ_API_KEY'] = 'gsk_...'
os.environ['SENDGRID_API_KEY'] = 'SG...'
os.environ['SENDGRID_FROM_EMAIL'] = 'your_email@domain.com'
os.environ['SENDGRID_TO_EMAIL'] = 'target_email@domain.com'
os.environ['NEWSDATA_API_KEY'] = 'pub_...'

print("Environment Variables Set!")

In [ ]:
import time, socket
import subprocess

def wait_for_port(port, timeout=40):
    start_time = time.time()
    while True:
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=1):
                return True
        except (OSError, ConnectionRefusedError):
            if time.time() - start_time > timeout:
                return False
            time.sleep(1)

print("🧹 Cleaning up existing processes...")
!fuser -k 8501/tcp

print("🚀 Launching AI Supply Chain Dashboard...")
get_ipython().system_raw("streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &")

if wait_for_port(8501):
    print("✅ Dashboard Backend Ready!")
    print("🌍 Initiating secure, zero-auth SSH Tunnel via Pinggy...")
    
    # Using Pinggy SSH tunnel (No login required, highly stable)
    # We redirect the output to a file so we can parse the URL
    subprocess.Popen(["ssh", "-o", "StrictHostKeyChecking=no", "-p", "443", "-R0:localhost:8501", "a.pinggy.io"], stdout=open('pinggy.log', 'w'))
    
    time.sleep(5) # Give it 5 seconds to establish the connection
    try:
        with open('pinggy.log', 'r') as f:
            log_lines = f.readlines()
            
            url = None
            for line in log_lines:
                if "http://" in line or "https://" in line:
                    url = line.strip()
                    break
            
            if url:
                print("\n--- 🔗 SECURE UNIVERSAL ACCESS (No Login) ---")
                print(f"Dashboard URL: {url}")
                print("----------------------------------------------\n")
            else:
                print("Tunnel connected. Run '!cat pinggy.log' to see the exact link.")
    except Exception as e:
        print("Failed to read tunnel output. Please check network.")
    
else:
    print("❌ Startup failed. Displaying logs:")
    !cat streamlit.log